# Análisis de los comentarios


In [ ]:
import pandas as pd
from pathlib import Path
import re
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import nltk
import numpy as np
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('stopwords')
nltk.download('punkt_tab')


### Carga de los datos

In [ ]:
project_root = Path().resolve().parents[1]
file = project_root/ "data" / "processed" / "steam_reviews_processed.parquet"
df = pd.read_parquet(file)

In [ ]:
stop_words = set(stopwords.words("english"))
def clean_text(text, stop_words):
    """Se queda solo lo que es texto"""
    text = text.split()
    text = " ".join(word for word in text if word not in stop_words)
    text = re.sub(r"[^a-z\s]", "", text)
    text = text.split()
    text = " ".join(text)
    return text

In [ ]:
def graph_wordcloud(text,stop_words,is_pos):
    colormaps = "summer" if is_pos else "autumn"
    wordcloud = WordCloud(width=800, height=400, max_words=100, collocations=True, stopwords=stop_words, background_color="white", min_font_size=10, colormap=colormaps, random_state=42).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")  
    plt.show()

### Separación de comentarios positivos y negativos

In [ ]:
df_positive = df[df["is_positive"]]
df_negative = df[~df["is_positive"]]

In [ ]:
pos_text = " ".join(df_positive["text"].astype(str).tolist())
pos_text = clean_text(pos_text, stop_words)

### Palabras en reviews positivas

In [ ]:
graph_wordcloud(pos_text,stop_words,True)

#### Limpio

In [ ]:
pos_stopwords = stop_words.copy()
# eliminar palabras filler e irrelevantes
pos_stopwords.update(["even", "one", "way", "make", "play", "see", "found", "thing", "youre", "item", "without", "also"]) 
graph_wordcloud(pos_text,pos_stopwords,True)

In [ ]:
neg_text = " ".join(df_negative["text"].astype(str).tolist())
neg_text = clean_text(neg_text, stop_words)

### Palabras en reviews negativas

In [ ]:
graph_wordcloud(neg_text,stop_words,False)

#### Limpio

In [ ]:
neg_stopwords = stop_words.copy()
# hay que quitar palabras trampa que probablemente sean "not good, not fun", etc.
neg_stopwords.update(["game", "one", "thing", "really","feel", "even", "good", "fun", "going", "way", "also", "play", "got", "put", "know", "thats","like", "get", "games", "make","heartsheartsheartshearts","much", "made", "want"])
graph_wordcloud(neg_text,neg_stopwords,False)

In [ ]:
pos_words = pos_text.split()
freq = Counter(pos_words)
pos_dict = {}
for word, counter in freq.most_common():
    pos_dict[word] = counter
pos_dict

In [ ]:
neg_words = neg_text.split()
freq2 = Counter(neg_words)
neg_dict = {}
for word, counter in freq2.most_common():
    neg_dict[word] = counter
neg_dict

### Análisis con TF-IDF

In [ ]:
pos_text = "\n".join(df_positive["text"].astype(str).tolist())
neg_text = "\n".join(df_negative["text"].astype(str).tolist())
pos_reviews = [clean_text(r,stop_words) for r in pos_text.split("\n") if r.strip()]
neg_reviews = [clean_text(r,stop_words) for r in neg_text.split("\n") if r.strip()]

documents = pos_reviews + neg_reviews
labels = np.array([1]*len(pos_reviews) + [0]*len(neg_reviews))

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_df=0.85, # ignorar los que estan en mas del 85% de documentos
    min_df=5, # ignorar los que estan en menos de 5 documentos
    max_features=5000 # numero de palabras consideradas
)

X = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

pos_mean = X[labels == 1].mean(axis=0).A1  # pos_mean[i] es la importancia media de palabra i en las reviews positivas
neg_mean = X[labels == 0].mean(axis=0).A1  # analogo a pos_mean

diff = pos_mean - neg_mean
print(diff)
top_pos = np.argsort(diff)[-100:] # indices que ordenarían el array diff
top_neg = np.argsort(diff)[:100]

print("\nTop palabras positivas:")
print([feature_names[i] for i in reversed(top_pos)])

print("\nTop palabras negativas:")
print([feature_names[i] for i in top_neg])

### Nube de palabras con TF-IDF

In [ ]:
top_positive_words = {feature_names[i]: diff[i] for i in reversed(top_pos) }
top_negative_words = {feature_names[i]: abs(diff[i]) for i in top_neg }

cloud_pos = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="summer",
    collocations = False
).generate_from_frequencies(top_positive_words)

plt.figure(figsize=(10,5))
plt.imshow(cloud_pos)
plt.axis("off")
plt.show()


cloud_neg = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="autumn",
    collocations = False
).generate_from_frequencies(top_negative_words)


plt.figure(figsize=(10,5))
plt.imshow(cloud_neg)
plt.axis("off")
plt.show()